### NBA Pregame Gameplan Report

The goal of this notebook is to create a coach facing report to prepare for a game. This report will include goals for the game that are key for the chosen team to win against the given opponent.

## 1. Imports

In [1876]:
from __future__ import annotations

from pathlib import Path
from io import BytesIO
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnnotationBbox

from PIL import Image

from matplotlib.backends.backend_pdf import PdfPages
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from nba_api.stats.endpoints import leaguegamelog
from nba_api.stats.static import teams

import urllib.request
import requests

## 2. Inputs

In [1877]:
SEASON = "2024-25"
SEASON_TYPE = "Regular Season"

MY_TEAM = "New York Knicks"
OPPONENT = "Boston Celtics"

NBA_TEAMS = teams.get_teams()
TEAM_ABBR_LOOKUP = {t["abbreviation"]: t for t in NBA_TEAMS}

MY_TEAM_ABBR = next(t["abbreviation"] for t in NBA_TEAMS if t["full_name"] == MY_TEAM)
OPPONENT_ABBR = next(t["abbreviation"] for t in NBA_TEAMS if t["full_name"] == OPPONENT)

TEAM_COLORS = {
    "ATL": ("#E03A3E", "#C1D32F", "https://en.wikipedia.org/wiki/Special:FilePath/Atlanta_Hawks_logo.svg?width=512"),
    "BOS": ("#007A33", "#BA9653", "https://en.wikipedia.org/wiki/Special:FilePath/Boston_Celtics.svg?width=512"),
    "BKN": ("#000000", "#FFFFFF", "https://en.wikipedia.org/wiki/Special:FilePath/Brooklyn_Nets_newlogo.svg?width=512"),
    "CHA": ("#1D1160", "#00788C", "https://en.wikipedia.org/wiki/Special:FilePath/Charlotte_Hornets_(2014).svg?width=512"),
    "CHI": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Chicago_Bulls_logo.svg?width=512"),
    "CLE": ("#860038", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Cleveland_Cavaliers_logo.svg?width=512"),
    "DAL": ("#00538C", "#002B5E", "https://en.wikipedia.org/wiki/Special:FilePath/Dallas_Mavericks_logo.svg?width=512"),
    "DEN": ("#0E2240", "#FEC524", "https://en.wikipedia.org/wiki/Special:FilePath/Denver_Nuggets.svg?width=512"),
    "DET": ("#C8102E", "#1D42BA", "https://en.wikipedia.org/wiki/Special:FilePath/Logo_of_the_Detroit_Pistons.svg?width=512"),
    "GSW": ("#1D428A", "#FFC72C", "https://en.wikipedia.org/wiki/Special:FilePath/Golden_State_Warriors_logo.svg?width=512"),
    "HOU": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Houston_Rockets.svg?width=512"),
    "IND": ("#002D62", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Indiana_Pacers.svg?width=512"),
    "LAC": ("#C8102E", "#1D428A", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Clippers_(2024).svg?width=512"),
    "LAL": ("#552583", "#FDB927", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Lakers_logo.svg?width=512"),
    "MEM": ("#5D76A9", "#12173F", "https://en.wikipedia.org/wiki/Special:FilePath/Memphis_Grizzlies.svg?width=512"),
    "MIA": ("#98002E", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Miami_Heat_logo.svg?width=512"),
    "MIL": ("#00471B", "#EEE1C6", "https://en.wikipedia.org/wiki/Special:FilePath/Milwaukee_Bucks_logo.svg?width=512"),
    "MIN": ("#0C2340", "#78BE20", "https://en.wikipedia.org/wiki/Special:FilePath/Minnesota_Timberwolves_logo.svg?width=512"),
    "NOP": ("#0C2340", "#C8102E", "https://en.wikipedia.org/wiki/Special:FilePath/New_Orleans_Pelicans_logo.svg?width=512"),
    "NYK": ("#006BB6", "#F58426", "https://en.wikipedia.org/wiki/Special:FilePath/New_York_Knicks_logo.svg?width=512"),
    "OKC": ("#007AC1", "#EF3B24", "https://en.wikipedia.org/wiki/Special:FilePath/Oklahoma_City_Thunder.svg?width=512"),
    "ORL": ("#0077C0", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Orlando_Magic_logo.svg?width=512"),
    "PHI": ("#006BB6", "#ED174C", "https://en.wikipedia.org/wiki/Special:FilePath/Philadelphia_76ers_logo.svg?width=512"),
    "PHX": ("#1D1160", "#E56020", "https://en.wikipedia.org/wiki/Special:FilePath/Phoenix_Suns_logo.svg?width=512"),
    "POR": ("#E03A3E", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Portland_Trail_Blazers_logo.svg?width=512"),
    "SAC": ("#5A2D81", "#63727A", "https://en.wikipedia.org/wiki/Special:FilePath/SacramentoKings.svg?width=512"),
    "SAS": ("#C4CED4", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/San_Antonio_Spurs.svg?width=512"),
    "TOR": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Toronto_Raptors_logo.svg?width=512"),
    "UTA": ("#002B5C", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Utah_Jazz_logo_2022.svg?width=512"),
    "WAS": ("#002B5C", "#E31837", "https://en.wikipedia.org/wiki/Special:FilePath/Washington_Wizards_logo.svg?width=512"),
}

OUTPUT_DIR = Path("pregame_gameplan_reports")
OUTPUT_DIR.mkdir(exist_ok=True)

REPORT_PATH = OUTPUT_DIR / f"{MY_TEAM_ABBR}_vs_{OPPONENT_ABBR}_{SEASON}_{SEASON_TYPE.replace(' ', '_')}_gameplan.pdf"

## 3. Load Team Game Logs

In [1878]:
def load_team_games(season: str, season_type: str) -> pd.DataFrame:
    logs = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        player_or_team_abbreviation="T",
    )
    return logs.get_data_frames()[0]


raw_games = load_team_games(SEASON, SEASON_TYPE)
raw_games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
1,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
2,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1
4,22024,1610612753,ORL,Orlando Magic,0022400065,2024-10-23,ORL @ MIA,W,240,41,...,39,57,28,8,8,14,21,116,19,1


## 4. Clean and Create Opponent Rows

In [1879]:
def prepare_games(raw: pd.DataFrame) -> pd.DataFrame:
    keep_cols = [
        "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME", "GAME_ID", "GAME_DATE",
        "MATCHUP", "WL", "MIN", "PTS", "FGM", "FGA", "FG3M", "FG3A",
        "FTM", "FTA", "OREB", "DREB", "REB", "AST", "STL", "BLK",
        "TOV", "PF", "PLUS_MINUS",
    ]

    df = raw[keep_cols].copy()
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    df["WIN"] = (df["WL"] == "W").astype(int)

    numeric_cols = [
        "MIN", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PLUS_MINUS",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["POSS"] = df["FGA"] + 0.44 * df["FTA"] - df["OREB"] + df["TOV"]

    opp = df[
        [
            "GAME_ID", "TEAM_ID", "TEAM_NAME", "TEAM_ABBREVIATION", "PTS",
            "FGA", "FG3A", "FTA", "OREB", "DREB", "TOV", "POSS", "PF",
        ]
    ].copy()

    opp = opp.rename(columns={
        "TEAM_ID": "OPP_TEAM_ID",
        "TEAM_NAME": "OPP_TEAM_NAME",
        "TEAM_ABBREVIATION": "OPP_TEAM_ABBREVIATION",
        "PTS": "OPP_PTS",
        "FGA": "OPP_FGA",
        "FG3A": "OPP_FG3A",
        "FTA": "OPP_FTA",
        "OREB": "OPP_OREB",
        "DREB": "OPP_DREB",
        "TOV": "OPP_TOV",
        "POSS": "OPP_POSS",
        "PF": "OPP_PF",
    })

    games = df.merge(opp, on="GAME_ID", how="left")
    games = games[games["TEAM_ID"] != games["OPP_TEAM_ID"]].copy()

    return games.sort_values(["GAME_DATE", "GAME_ID", "TEAM_NAME"]).reset_index(drop=True)


games = prepare_games(raw_games)

## 5. Same Game Stats for Labels and Future Goal Evaluation

In [1880]:
def add_actual_game_stats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ACT_OFF_RTG"] = 100 * out["PTS"] / out["POSS"]
    out["ACT_DEF_RTG"] = 100 * out["OPP_PTS"] / out["OPP_POSS"]
    out["ACT_NET_RTG"] = out["ACT_OFF_RTG"] - out["ACT_DEF_RTG"]

    out["ACT_TOV_PCT"] = 100 * out["TOV"] / out["POSS"]
    out["ACT_OPP_TOV_PCT"] = 100 * out["OPP_TOV"] / out["OPP_POSS"]

    out["ACT_OREB_PCT"] = 100 * out["OREB"] / (out["OREB"] + out["OPP_DREB"])
    out["ACT_DREB_PCT"] = 100 * out["DREB"] / (out["DREB"] + out["OPP_OREB"])

    out["ACT_FTA_RATE"] = out["FTA"] / out["FGA"]
    out["ACT_OPP_FTA_RATE"] = out["OPP_FTA"] / out["OPP_FGA"]

    out["ACT_3PA_RATE"] = 100 * out["FG3A"] / out["FGA"]
    out["ACT_OPP_3PA_RATE"] = 100 * out["OPP_FG3A"] / out["OPP_FGA"]

    out["ACT_PACE"] = out[["POSS", "OPP_POSS"]].mean(axis=1) * 48 / (out["MIN"] / 5)

    return out.replace([np.inf, -np.inf], np.nan)


games = add_actual_game_stats(games)

## 6. Build Pregame Rolling Features

In [1881]:
ROLLING_BASE_COLS = [
    "ACT_OFF_RTG",
    "ACT_DEF_RTG",
    "ACT_NET_RTG",
    "ACT_TOV_PCT",
    "ACT_OREB_PCT",
    "ACT_DREB_PCT",
    "ACT_FTA_RATE",
    "ACT_3PA_RATE",
    "ACT_PACE",
]


def add_team_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["TEAM_ID", "GAME_DATE", "GAME_ID"]).copy()

    for col in ROLLING_BASE_COLS:
        out[f"TEAM_SEASON_{col}"] = (
            out.groupby("TEAM_ID")[col]
            .transform(lambda s: s.shift(1).expanding().mean())
        )

        out[f"TEAM_L5_{col}"] = (
            out.groupby("TEAM_ID")[col]
            .transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())
        )

        out[f"TEAM_L10_{col}"] = (
            out.groupby("TEAM_ID")[col]
            .transform(lambda s: s.shift(1).rolling(10, min_periods=5).mean())
        )

    return out


rolling_games = add_team_rolling_features(games)

## 7. Attach Opponent Pregame Rolling Features

In [1882]:
def attach_opponent_pregame_features(df: pd.DataFrame) -> pd.DataFrame:
    team_future_cols = [
        col for col in df.columns
        if col.startswith("TEAM_SEASON_") or col.startswith("TEAM_L5_") or col.startswith("TEAM_L10_")
    ]
    opp_features = df[
        ["GAME_ID", "TEAM_ID"] + team_future_cols
    ].copy()
    opp_features = opp_features.rename(columns={"TEAM_ID": "OPP_TEAM_ID"})
    rename_map = {
        col: col.replace("TEAM_", "OPP_")
        for col in team_future_cols
    }
    opp_features = opp_features.rename(columns=rename_map)
    out = df.merge(opp_features, on=["GAME_ID", "OPP_TEAM_ID"], how="left")
    return out

pregame_df = attach_opponent_pregame_features(rolling_games)

## 8. Add Matchup Dependent Difference Features

In [1883]:
def add_matchup_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["PREGAME_NET_RTG_DIFF"] = out["TEAM_SEASON_ACT_NET_RTG"] - out["OPP_SEASON_ACT_NET_RTG"]
    out["PREGAME_L5_NET_RTG_DIFF"] = out["TEAM_L5_ACT_NET_RTG"] - out["OPP_L5_ACT_NET_RTG"]
    out["PREGAME_L10_NET_RTG_DIFF"] = out["TEAM_L10_ACT_NET_RTG"] - out["OPP_L10_ACT_NET_RTG"]

    out["PREGAME_OFF_VS_DEF"] = out["TEAM_SEASON_ACT_OFF_RTG"] - out["OPP_SEASON_ACT_DEF_RTG"]
    out["PREGAME_DEF_VS_OFF"] = out["OPP_SEASON_ACT_OFF_RTG"] - out["TEAM_SEASON_ACT_DEF_RTG"]

    out["PREGAME_TOV_EDGE"] = out["OPP_SEASON_ACT_TOV_PCT"] - out["TEAM_SEASON_ACT_TOV_PCT"]
    out["PREGAME_OREB_EDGE"] = out["TEAM_SEASON_ACT_OREB_PCT"] - (100 - out["OPP_SEASON_ACT_DREB_PCT"])
    out["PREGAME_FTA_EDGE"] = out["TEAM_SEASON_ACT_FTA_RATE"] - out["OPP_SEASON_ACT_FTA_RATE"]
    out["PREGAME_3PA_STYLE_DIFF"] = out["TEAM_SEASON_ACT_3PA_RATE"] - out["OPP_SEASON_ACT_3PA_RATE"]
    out["PREGAME_PACE_DIFF"] = out["TEAM_SEASON_ACT_PACE"] - out["OPP_SEASON_ACT_PACE"]

    return out

pregame_df = add_matchup_features(pregame_df)

## 9. Add Previous Head-to-Head Features

In [1884]:
def add_head_to_head_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["TEAM_ID", "OPP_TEAM_ID", "GAME_DATE", "GAME_ID"]).copy()
    out["H2H_PREV_GAMES"] = (
        out.groupby(["TEAM_ID", "OPP_TEAM_ID"]).cumcount()
    )
    out["H2H_PREV_WIN_PCT"] = (
        out.groupby(["TEAM_ID", "OPP_TEAM_ID"])["WIN"].transform(lambda s: s.shift(1).expanding().mean())
    )
    out["H2H_PREV_NET_RTG"] = (
        out.groupby(["TEAM_ID", "OPP_TEAM_ID"])["ACT_NET_RTG"].transform(lambda s: s.shift(1).expanding().mean())
    )
    out["H2H_PREV_WIN_PCT"] = out["H2H_PREV_WIN_PCT"].fillna(0.5)
    out["H2H_PREV_NET_RTG"] = out["H2H_PREV_NET_RTG"].fillna(0.0)
    return out

pregame_df = add_head_to_head_features(pregame_df)

## 10. Train True Pregame Model

In [1885]:
pregame_features = [
    "PREGAME_NET_RTG_DIFF",
    "PREGAME_L5_NET_RTG_DIFF",
    "PREGAME_L10_NET_RTG_DIFF",
    "PREGAME_OFF_VS_DEF",
    "PREGAME_DEF_VS_OFF",
    "PREGAME_TOV_EDGE",
    "PREGAME_OREB_EDGE",
    "PREGAME_FTA_EDGE",
    "PREGAME_3PA_STYLE_DIFF",
    "PREGAME_PACE_DIFF",
    "H2H_PREV_GAMES",
    "H2H_PREV_WIN_PCT",
    "H2H_PREV_NET_RTG",
]

model_data = pregame_df.dropna(subset=pregame_features + ["WIN"]).copy()

X = model_data[pregame_features]
y = model_data["WIN"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42, stratify = y)

pregame_model = Pipeline(steps = [
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

pregame_model.fit(X_train, y_train)
pred = pregame_model.predict(X_test)
proba = pregame_model.predict_proba(X_test)[:, 1]

print("Pregame Model Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Pregame Model AUC:", round(roc_auc_score(y_test, proba), 3))

Pregame Model Accuracy: 0.658
Pregame Model AUC: 0.749


/Users/gregorylederer/Desktop/NBA_Samples/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gregorylederer/Desktop/NBA_Samples/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gregorylederer/Desktop/NBA_Samples/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gregorylederer/Desktop/NBA_Samples/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/gregorylederer/Desktop/NBA_Samples/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: Runti

## 11. Model Signals

In [1886]:
model_signals = pd.DataFrame({
    "feature": pregame_features,
    "coefficient": pregame_model.named_steps["model"].coef_[0],
})
model_signals["abs_importance"] = model_signals["coefficient"].abs()
model_signals = model_signals.sort_values("abs_importance", ascending=False)
model_signals

,feature,coefficient,abs_importance
0,PREGAME_NET_RTG_DIFF,0.514909,0.514909
11,H2H_PREV_WIN_PCT,0.246967,0.246967
2,PREGAME_L10_NET_RTG_DIFF,0.246635,0.246635
5,PREGAME_TOV_EDGE,0.131506,0.131506
12,H2H_PREV_NET_RTG,-0.125701,0.125701
8,PREGAME_3PA_STYLE_DIFF,-0.119857,0.119857
1,PREGAME_L5_NET_RTG_DIFF,0.082395,0.082395
4,PREGAME_DEF_VS_OFF,0.073852,0.073852
6,PREGAME_OREB_EDGE,-0.060058,0.060058
9,PREGAME_PACE_DIFF,0.059580,0.059580


## 12. Get Latest Pregame Match Profile

In [1887]:
def latest_team_row(df: pd.DataFrame, team_name: str) -> pd.Series:
    team_rows = df[df["TEAM_NAME"] == team_name].sort_values("GAME_DATE")
    return team_rows.iloc[-1]

def build_current_matchup_profile(df: pd.DataFrame, my_team: str, opponent: str) -> pd.Series:
    my_latest = latest_team_row(df, my_team)
    opp_latest = latest_team_row(df, opponent)
    row = my_latest.copy()
    for col in df.columns:
        if col.startswith("TEAM_SEASON_") or col.startswith("TEAM_L5_") or col.startswith("TEAM_L10_"):
            row[col.replace("TEAM_", "OPP_")] = opp_latest[col]
    row["OPP_TEAM_NAME"] = opponent
    row = add_matchup_features(pd.DataFrame([row])).iloc[0]
    h2h = df[
        (df["TEAM_NAME"] == my_team) &
        (df["OPP_TEAM_NAME"] == opponent)
    ].sort_values("GAME_DATE")
    row["H2H_PREV_GAMES"] = len(h2h)
    row["H2H_PREV_WIN_PCT"] = h2h["WIN"].mean() if len(h2h) else 0.5
    row["H2H_PREV_NET_RTG"] = h2h["ACT_NET_RTG"].mean() if len(h2h) else 0
    return row

matchup = build_current_matchup_profile(pregame_df, MY_TEAM, OPPONENT)
matchup_win_prob = pregame_model.predict_proba(
    pd.DataFrame([matchup[pregame_features]])
)[0,1]

print(f"{MY_TEAM} pregame win probability vs {OPPONENT}: {matchup_win_prob:.1%}")

New York Knicks pregame win probability vs Boston Celtics: 36.9%


## 13. Matchup Specific Goal Model & Evidence Tables

In [1888]:
GOAL_CONFIG = [
    {
        "goal": "Offensive Efficiency",
        "metric": "ORtg",
        "target_stat": "ACT_OFF_RTG",
        "team_avg_col": "TEAM_SEASON_ACT_OFF_RTG",
        "team_l5_col": "TEAM_L5_ACT_OFF_RTG",
        "team_l10_col": "TEAM_L10_ACT_OFF_RTG",
        "opp_context_col": "OPP_SEASON_ACT_DEF_RTG",
        "rank_stat": "TEAM_SEASON_ACT_OFF_RTG",
        "signal_feature": "PREGAME_OFF_VS_DEF",
        "better": "higher",
        "target_fmt": "{:.1f}+",
    },
    {
        "goal": "Defensive Efficiency",
        "metric": "DRtg",
        "target_stat": "ACT_DEF_RTG",
        "team_avg_col": "TEAM_SEASON_ACT_DEF_RTG",
        "team_l5_col": "TEAM_L5_ACT_DEF_RTG",
        "team_l10_col": "TEAM_L10_ACT_DEF_RTG",
        "opp_context_col": "OPP_SEASON_ACT_OFF_RTG",
        "rank_stat": "TEAM_SEASON_ACT_DEF_RTG",
        "signal_feature": "PREGAME_DEF_VS_OFF",
        "better": "lower",
        "target_fmt": "{:.1f}-",
    },
    {
        "goal": "Turnover Control",
        "metric": "TOV%",
        "target_stat": "ACT_TOV_PCT",
        "team_avg_col": "TEAM_SEASON_ACT_TOV_PCT",
        "team_l5_col": "TEAM_L5_ACT_TOV_PCT",
        "team_l10_col": "TEAM_L10_ACT_TOV_PCT",
        "opp_context_col": "OPP_SEASON_ACT_TOV_PCT",
        "rank_stat": "TEAM_SEASON_ACT_TOV_PCT",
        "signal_feature": "PREGAME_TOV_EDGE",
        "better": "lower",
        "target_fmt": "{:.1f}%-",
    },
    {
        "goal": "Offensive Glass",
        "metric": "OREB%",
        "target_stat": "ACT_OREB_PCT",
        "team_avg_col": "TEAM_SEASON_ACT_OREB_PCT",
        "team_l5_col": "TEAM_L5_ACT_OREB_PCT",
        "team_l10_col": "TEAM_L10_ACT_OREB_PCT",
        "opp_context_col": "OPP_SEASON_ACT_DREB_PCT",
        "rank_stat": "TEAM_SEASON_ACT_OREB_PCT",
        "signal_feature": "PREGAME_OREB_EDGE",
        "better": "higher",
        "target_fmt": "{:.1f}%+",
    },
    {
        "goal": "Free Throw Pressure",
        "metric": "FTA/FGA",
        "target_stat": "ACT_FTA_RATE",
        "team_avg_col": "TEAM_SEASON_ACT_FTA_RATE",
        "team_l5_col": "TEAM_L5_ACT_FTA_RATE",
        "team_l10_col": "TEAM_L10_ACT_FTA_RATE",
        "opp_context_col": "OPP_SEASON_ACT_FTA_RATE",
        "rank_stat": "TEAM_SEASON_ACT_FTA_RATE",
        "signal_feature": "PREGAME_FTA_EDGE",
        "better": "higher",
        "target_fmt": "{:.3f}+",
    },
    {
        "goal": "3PT Volume",
        "metric": "3PA Rate",
        "target_stat": "ACT_3PA_RATE",
        "team_avg_col": "TEAM_SEASON_ACT_3PA_RATE",
        "team_l5_col": "TEAM_L5_ACT_3PA_RATE",
        "team_l10_col": "TEAM_L10_ACT_3PA_RATE",
        "opp_context_col": "OPP_SEASON_ACT_3PA_RATE",
        "rank_stat": "TEAM_SEASON_ACT_3PA_RATE",
        "signal_feature": "PREGAME_3PA_STYLE_DIFF",
        "better": "higher",
        "target_fmt": "{:.1f}%+",
    },
    {
        "goal": "Game Tempo",
        "metric": "Pace",
        "target_stat": "ACT_PACE",
        "team_avg_col": "TEAM_SEASON_ACT_PACE",
        "team_l5_col": "TEAM_L5_ACT_PACE",
        "team_l10_col": "TEAM_L10_ACT_PACE",
        "opp_context_col": "OPP_SEASON_ACT_PACE",
        "rank_stat": "TEAM_SEASON_ACT_PACE",
        "signal_feature": "PREGAME_PACE_DIFF",
        "better": "range",
        "target_fmt": "{:.1f}",
    },
]


def latest_profiles_by_team(df: pd.DataFrame) -> pd.DataFrame:
    latest = (
        df.sort_values(["TEAM_ID", "GAME_DATE", "GAME_ID"])
        .groupby("TEAM_ID")
        .tail(1)
        .copy()
    )
    return latest.reset_index(drop=True)


def add_rank_columns(profile_df: pd.DataFrame) -> pd.DataFrame:
    out = profile_df.copy()

    rank_rules = {
        "TEAM_SEASON_ACT_OFF_RTG": False,
        "TEAM_SEASON_ACT_DEF_RTG": True,
        "TEAM_SEASON_ACT_NET_RTG": False,
        "TEAM_SEASON_ACT_TOV_PCT": True,
        "TEAM_SEASON_ACT_OREB_PCT": False,
        "TEAM_SEASON_ACT_DREB_PCT": False,
        "TEAM_SEASON_ACT_FTA_RATE": False,
        "TEAM_SEASON_ACT_3PA_RATE": False,
        "TEAM_SEASON_ACT_PACE": False,
    }

    for col, ascending in rank_rules.items():
        out[f"{col}_RANK"] = out[col].rank(ascending=ascending, method="min").astype(int)

    return out


team_profiles = add_rank_columns(latest_profiles_by_team(pregame_df))


def get_team_record(df: pd.DataFrame, team_name: str) -> str:
    team_games = df[df["TEAM_NAME"] == team_name]
    wins = int(team_games["WIN"].sum())
    losses = int(len(team_games) - wins)
    return f"{wins}-{losses}"


def value_rank(team_name: str, col: str) -> str:
    row = team_profiles.loc[team_profiles["TEAM_NAME"] == team_name].iloc[0]
    rank = int(row[f"{col}_RANK"])
    return f"{rank}/30"


def format_value(value: float, metric: str) -> str:
    if pd.isna(value):
        return "--"
    if metric in ["FTA/FGA"]:
        return f"{value:.3f}"
    if metric in ["TOV%", "OREB%", "3PA Rate"]:
        return f"{value:.1f}%"
    return f"{value:.1f}"


def target_from_context(df: pd.DataFrame, matchup: pd.Series, cfg: dict) -> float:
    stat = cfg["target_stat"]
    better = cfg["better"]

    team_avg = matchup[cfg["team_avg_col"]]
    team_l5 = matchup[cfg["team_l5_col"]]
    team_l10 = matchup[cfg["team_l10_col"]]
    opp_context = matchup[cfg["opp_context_col"]]

    if better == "higher":
        win_threshold = df[df["WIN"] == 1][stat].quantile(0.62)
        target = np.nanmean([team_avg, team_l5, team_l10, win_threshold])

        if cfg["goal"] == "Offensive Glass":
            opp_oreb_allowed = 100 - opp_context
            target = np.nanmean([team_avg, team_l5, team_l10, win_threshold, opp_oreb_allowed + 1.0])

        return target

    if better == "lower":
        win_threshold = df[df["WIN"] == 1][stat].quantile(0.38)
        return np.nanmean([team_avg, team_l5, team_l10, win_threshold])

    return np.nanmean([team_avg, team_l5, team_l10, opp_context])


def normalized_goal_scores(goals: pd.DataFrame) -> pd.Series:
    raw = goals["Raw Score"].astype(float)
    if raw.max() == raw.min():
        return pd.Series([75] * len(raw), index=goals.index)
    return 55 + 40 * (raw - raw.min()) / (raw.max() - raw.min())


def build_matchup_goals_v2(df: pd.DataFrame, matchup: pd.Series, model_signals: pd.DataFrame) -> pd.DataFrame:
    coef_lookup = model_signals.set_index("feature")["abs_importance"].to_dict()
    max_coef = max(coef_lookup.values()) if coef_lookup else 1

    rows = []

    for cfg in GOAL_CONFIG:
        target = target_from_context(df, matchup, cfg)

        if cfg["better"] == "range":
            low = target - 1.7
            high = target + 1.7
            target_label = f"{low:.1f} - {high:.1f}"
            gap = abs(matchup[cfg["team_avg_col"]] - matchup[cfg["opp_context_col"]])
        else:
            target_label = cfg["target_fmt"].format(target)
            gap = abs(target - matchup[cfg["team_avg_col"]])

        model_strength = coef_lookup.get(cfg["signal_feature"], 0) / max_coef
        matchup_edge = abs(matchup.get(cfg["signal_feature"], 0))
        raw_score = (0.65 * model_strength) + (0.35 * matchup_edge / 10)

        rows.append({
            "Goal": cfg["goal"],
            "Metric": cfg["metric"],
            "Target": target_label,
            "Season": format_value(matchup[cfg["team_avg_col"]], cfg["metric"]),
            "L5": format_value(matchup[cfg["team_l5_col"]], cfg["metric"]),
            "L10": format_value(matchup[cfg["team_l10_col"]], cfg["metric"]),
            "Opp Ctx": format_value(matchup[cfg["opp_context_col"]], cfg["metric"]),
            "Rank": value_rank(matchup["TEAM_NAME"], cfg["rank_stat"]),
            "Signal": cfg["signal_feature"].replace("PREGAME_", "").replace("_", " "),
            "Raw Score": raw_score,
        })

    goals = pd.DataFrame(rows)
    goals["Importance"] = normalized_goal_scores(goals).round(0).astype(int)
    goals = goals.sort_values("Importance", ascending=False).reset_index(drop=True)

    return goals[
        ["Goal", "Metric", "Target", "Importance", "Season", "L5", "L10", "Opp Ctx", "Rank", "Signal"]
    ]

gameplan_goals = build_matchup_goals_v2(pregame_df, matchup, model_signals)
gameplan_goals

,Goal,Metric,Target,Importance,Season,L5,L10,Opp Ctx,Rank,Signal
0,3PT Volume,3PA Rate,39.7%+,95,38.0%,38.9%,37.7%,53.6%,28/30,3PA STYLE DIFF
1,Defensive Efficiency,DRtg,109.6-,71,111.9,111.6,112.6,118.1,16/30,DEF VS OFF
2,Offensive Efficiency,ORtg,115.3+,70,114.9,112.6,112.7,108.3,5/30,OFF VS DEF
3,Turnover Control,TOV%,13.2%-,65,13.2%,13.7%,13.9%,11.9%,7/30,TOV EDGE
4,Offensive Glass,OREB%,26.4%+,61,25.7%,25.7%,27.2%,76.1%,13/30,OREB EDGE
5,Game Tempo,Pace,96.0 - 99.4,59,99.3,96.5,96.5,98.4,26/30,PACE DIFF
6,Free Throw Pressure,FTA/FGA,0.235+,55,0.235,0.218,0.218,0.215,24/30,FTA EDGE


## 14. 1 Page Landscape Report Design

In [1889]:
TEAM_PRIMARY, TEAM_SECONDARY, TEAM_LOGO = TEAM_COLORS.get(MY_TEAM_ABBR, ("#111827", "#F9FAFB", None))
OPP_PRIMARY, OPP_SECONDARY, OPP_LOGO = TEAM_COLORS.get(OPPONENT_ABBR, ("#111827", "#F9FAFB", None))

PAGE_BG = "#F5F6F8"
INK = "#111827"
MUTED = "#6B7280"
SOFT_MUTED = "#9CA3AF"
LINE = "#D9DEE7"
WHITE = "#FFFFFF"

ESPN_LOGO_ABBR = {
    "ATL": "atl", "BOS": "bos", "BKN": "bkn", "CHA": "cha", "CHI": "chi",
    "CLE": "cle", "DAL": "dal", "DEN": "den", "DET": "det", "GSW": "gs",
    "HOU": "hou", "IND": "ind", "LAC": "lac", "LAL": "lal", "MEM": "mem",
    "MIA": "mia", "MIL": "mil", "MIN": "min", "NOP": "no", "NYK": "ny",
    "OKC": "okc", "ORL": "orl", "PHI": "phi", "PHX": "phx", "POR": "por",
    "SAC": "sac", "SAS": "sa", "TOR": "tor", "UTA": "utah", "WAS": "wsh",
}

TEAM_CONFERENCE = {
    "ATL": "East", "BOS": "East", "BKN": "East", "CHA": "East", "CHI": "East",
    "CLE": "East", "DET": "East", "IND": "East", "MIA": "East", "MIL": "East",
    "NYK": "East", "ORL": "East", "PHI": "East", "TOR": "East", "WAS": "East",
    "DAL": "West", "DEN": "West", "GSW": "West", "HOU": "West", "LAC": "West",
    "LAL": "West", "MEM": "West", "MIN": "West", "NOP": "West", "OKC": "West",
    "PHX": "West", "POR": "West", "SAC": "West", "SAS": "West", "UTA": "West",
}

def load_logo_image(url):
    if not url:
        return None
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as response:
            return Image.open(BytesIO(response.read())).convert("RGBA")
    except Exception as e:
        print(f"Logo unavailable: {e}")
        return None
    
team_logo_img = load_logo_image(TEAM_LOGO)
opp_logo_img = load_logo_image(OPP_LOGO)

def display_team_name(team_name: str) -> str:
    return team_name.split()[-1]


def draw_rect(ax, x, y, w, h, color=WHITE, edge=LINE, lw=0.8, radius=0.012, zorder=1):
    box = patches.FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle=f"round,pad=0.004,rounding_size={radius}",
        linewidth=lw,
        edgecolor=edge,
        facecolor=color,
        zorder=zorder,
    )
    ax.add_patch(box)
    return box


def draw_team_card_rect(ax, x, y, w, h, color=WHITE, edge=LINE, lw=0.9, zorder=1):
    box = patches.Rectangle(
        (x, y),
        w,
        h,
        linewidth=lw,
        edgecolor=edge,
        facecolor=color,
        zorder=zorder,
    )
    ax.add_patch(box)
    return box


def draw_info_box(ax, x, y, w, h, label, value, value_color=INK):
    draw_rect(
        ax,
        x,
        y,
        w,
        h,
        color="#F8FAFC",
        edge="#E5E7EB",
        lw=0.45,
        radius=0.006,
        zorder=4,
    )

    ax.text(
        x + 0.007,
        y + h * 0.75,
        label.upper(),
        ha="left",
        va="center",
        fontsize=7,
        color=MUTED,
        weight="bold",
        zorder=5,
    )

    ax.text(
        x + 0.007,
        y + h * 0.22,
        value,
        ha="left",
        va="center",
        fontsize=9,
        color=value_color,
        weight="bold",
        zorder=5,
    )


def draw_metric_box(ax, x, y, w, h, label, value, rank, primary):
    draw_rect(
        ax,
        x,
        y,
        w,
        h,
        color="#F8FAFC",
        edge="#E5E7EB",
        lw=0.45,
        radius=0.006,
        zorder=4,
    )

    ax.text(
        x + 0.008,
        y + h * 0.71,
        label,
        ha="left",
        va="center",
        fontsize=9,
        color=MUTED,
        weight="bold",
        zorder=5,
    )

    ax.text(
        x + 0.008,
        y + h * 0.26,
        f"{value:.1f}",
        ha="left",
        va="center",
        fontsize=10.0,
        color=INK,
        weight="bold",
        zorder=5,
    )

    ax.text(
        x + 0.085,
        y + h * 0.24,
        f"Rk {int(rank)}",
        ha="right",
        va="center",
        fontsize=8.5,
        color=primary,
        weight="bold",
        zorder=5,
    )


def get_conference_label(team_name: str) -> str:
    standing = get_standing_summary(team_name)
    return "ECF" if standing["Conference"] == "East" else "WCF"


def draw_team_header_block(fig, ax, x, y, w, h, team_name, abbr, primary, secondary, profile_row, logo_img_select):
    recent = get_recent_summary(pregame_df, team_name)
    standing = get_standing_summary(team_name)

    draw_team_card_rect(ax, x, y, w, h, color=WHITE, edge=LINE, lw=0.9, zorder=1)

    bar_h = 0.018
    ax.add_patch(
        patches.Rectangle(
            (x, y + h - bar_h),
            w,
            bar_h,
            facecolor=primary,
            edgecolor="none",
            zorder=3,
        )
    )

    if logo_img_select is not None:
        logo_ax = fig.add_axes([x-0.007, y+0.028, 0.135, 0.075])
        logo_ax.axis("off")
        logo_ax.imshow(logo_img_select)
    else:
        ax.text(
            0.975, 0.50, abbr,
            color="white", fontsize=22, fontweight="bold",
            va="center", ha="right", alpha=0.95,
        )

    content_x = x + 0.132
    content_w = w - 0.154

    ax.text(
        content_x - 0.004,
        y + h - 0.039,
        team_name,
        ha="left",
        va="center",
        fontsize=14.2,
        weight="bold",
        color=INK,
        zorder=5,
    )

    top_y = y + h - 0.088
    box_h = 0.03
    gap = 0.02
    box_w = (content_w - 3 * gap) / 4

    info_boxes = [
        ("Record", recent["Record"], INK),
        ("Last 10", recent["Last 10"], INK),
        (f"{get_conference_label(team_name)} Rank", f"{standing['Conf Rank']}", primary),
        ("NBA Rank", f"{standing['NBA Rank']}", primary),
    ]

    for i, (label, value, value_color) in enumerate(info_boxes):
        draw_info_box(
            ax,
            content_x + i * (box_w + gap),
            top_y,
            box_w,
            box_h,
            label,
            value,
            value_color=value_color,
        )

    metric_y = y + 0.015
    metric_h = 0.04
    metric_gap = 0.018
    metric_w = (content_w - 2 * metric_gap) / 3

    metric_specs = [
        ("ORTG", profile_row["TEAM_SEASON_ACT_OFF_RTG"], profile_row["TEAM_SEASON_ACT_OFF_RTG_RANK"]),
        ("DRTG", profile_row["TEAM_SEASON_ACT_DEF_RTG"], profile_row["TEAM_SEASON_ACT_DEF_RTG_RANK"]),
        ("Pace", profile_row["TEAM_SEASON_ACT_PACE"], profile_row["TEAM_SEASON_ACT_PACE_RANK"]),
    ]

    for i, (label, value, rank) in enumerate(metric_specs):
        draw_metric_box(
            ax,
            content_x + i * (metric_w + metric_gap),
            metric_y,
            metric_w,
            metric_h,
            label,
            value,
            rank,
            primary,
        )

def get_team_record(df: pd.DataFrame, team_name: str) -> str:
    team_games = df[df["TEAM_NAME"] == team_name]
    wins = int(team_games["WIN"].sum())
    losses = int(len(team_games) - wins)
    return f"{wins}-{losses}"


def get_recent_summary(df: pd.DataFrame, team_name: str) -> dict:
    team_games = df[df["TEAM_NAME"] == team_name].sort_values(["GAME_DATE", "GAME_ID"]).copy()

    last10 = team_games.tail(10)
    last10_wins = int(last10["WIN"].sum())
    last10_losses = int(len(last10) - last10_wins)

    if len(team_games) == 0:
        streak = "--"

    else:
        wins = team_games["WIN"].tolist()
        last_result = wins[-1]
        count = 0

        for result in reversed(wins):
            if result == last_result:
                count += 1
            else:
                break

        streak = f"{'W' if last_result == 1 else 'L'}{count}"

    return {
        "Record": get_team_record(df, team_name),
        "Last 10": f"{last10_wins}-{last10_losses}",
        "Streak": streak,
    }


def build_standings_table(df: pd.DataFrame) -> pd.DataFrame:
    standings = (
        df.groupby(["TEAM_NAME", "TEAM_ABBREVIATION"], as_index=False)
        .agg(Wins=("WIN", "sum"), Games=("WIN", "count"))
    )

    standings["Losses"] = standings["Games"] - standings["Wins"]
    standings["WinPct"] = standings["Wins"] / standings["Games"]
    standings["Conference"] = standings["TEAM_ABBREVIATION"].map(TEAM_CONFERENCE)

    standings = standings.sort_values(["WinPct", "Wins"], ascending=False).reset_index(drop=True)
    standings["NBA Rank"] = np.arange(1, len(standings) + 1)

    standings["Conf Rank"] = (
        standings.sort_values(["Conference", "WinPct", "Wins"], ascending=[True, False, False])
        .groupby("Conference")
        .cumcount()
        + 1
    )

    return standings


STANDINGS = build_standings_table(pregame_df)


def get_standing_summary(team_name: str) -> dict:
    row = STANDINGS[STANDINGS["TEAM_NAME"] == team_name].iloc[0]
    return {
        "Conference": row["Conference"],
        "Conf Rank": int(row["Conf Rank"]),
        "NBA Rank": int(row["NBA Rank"]),
    }


def format_value(value: float, metric: str) -> str:
    if pd.isna(value):
        return "--"
    if metric in ["FTA/FGA"]:
        return f"{value:.3f}"
    if metric in ["TOV%", "OREB%", "3PA Rate"]:
        return f"{value:.1f}%"
    return f"{value:.1f}"

def draw_title_bar(ax):
    title_font = {
        "fontfamily": "DejaVu Sans",
        "fontweight": 800,
    }

    ax.text(
        0.035,
        0.940,
        f"{MY_TEAM_ABBR} - PREGAME GAMEPLAN",
        fontsize=10,
        color=MUTED,
        va="center",
        **title_font,
    )

    matchup_center_x = 0.5

    ax.text(
        matchup_center_x,
        0.958,
        MY_TEAM,
        fontsize=25,
        color=TEAM_PRIMARY,
        ha="center",
        va="center",
        **title_font,
    )

    ax.text(
        matchup_center_x,
        0.922,
        f"vs {OPPONENT}",
        fontsize=18,
        color=OPP_PRIMARY,
        ha="center",
        va="center",
        **title_font,
    )

    ax.text(
        0.860,
        0.945,
        "PREGAME WIN PROBABILITY",
        fontsize=10,
        fontfamily="DejaVu Sans",
        fontweight=800,
        color=MUTED,
        ha="right",
        va="center",
    )

    ax.text(
        0.860,
        0.928,
        MY_TEAM.upper(),
        fontsize=10,
        fontfamily="DejaVu Sans",
        fontweight=800,
        color=MUTED,
        ha="right",
        va="center",
    )

    draw_rect(
        ax,
        0.875,
        0.915,
        0.090,
        0.052,
        color=TEAM_PRIMARY,
        edge=TEAM_PRIMARY,
        lw=0.0,
        radius=0.014,
        zorder=4,
    )

    ax.text(
        0.920,
        0.940,
        f"{matchup_win_prob:.0%}",
        fontsize=22,
        fontfamily="DejaVu Sans",
        fontweight=800,
        color="white",
        ha="center",
        va="center",
        zorder=5,
    )


def draw_goal_cards(ax, goals: pd.DataFrame):
    top_goals = goals.head(7).copy()

    start_x = 0.035
    y = 0.505
    gap = 0.008
    card_w = (0.93 - 6 * gap) / 7
    card_h = 0.150

    for i, row in top_goals.iterrows():
        x = start_x + i * (card_w + gap)

        ax.add_patch(
            patches.Rectangle(
                (x, y),
                card_w,
                card_h,
                linewidth=0.8,
                edgecolor=LINE,
                facecolor=WHITE,
                zorder=1,
            )
        )

        ax.add_patch(
            patches.Rectangle(
                (x, y + card_h - 0.012),
                card_w,
                0.012,
                facecolor=TEAM_PRIMARY,
                edgecolor="none",
                zorder=4,
            )
        )

        ax.text(
            x + 0.010,
            y + card_h - 0.031,
            row["Goal"],
            fontsize=9,
            color=INK,
            weight="bold",
            va="center",
            zorder=5,
        )

        ax.text(
            x + 0.010,
            y + card_h - 0.056,
            row["Metric"],
            fontsize=8,
            color=MUTED,
            weight="bold",
            va="center",
            zorder=5,
        )

        ax.text(
            x + 0.010,
            y + 0.067,
            row["Target"],
            fontsize=15,
            color=TEAM_PRIMARY,
            weight="bold",
            va="center",
            zorder=5,
        )

        ax.text(
            x + 0.010,
            y + 0.035,
            "IMPORTANCE",
            fontsize=8,
            color=MUTED,
            weight="bold",
            zorder=5,
        )

        ax.text(
            x + card_w - 0.010,
            y + 0.035,
            f"{row['Importance']}/100",
            fontsize=8,
            color=INK,
            weight="bold",
            ha="right",
            zorder=5,
        )

        bar_x = x + 0.010
        bar_y = y + 0.017
        bar_w = card_w - 0.020

        ax.add_patch(
            patches.Rectangle(
                (bar_x, bar_y),
                bar_w,
                0.008,
                facecolor="#E5E7EB",
                edgecolor="none",
                zorder=4,
            )
        )

        ax.add_patch(
            patches.Rectangle(
                (bar_x, bar_y),
                bar_w * row["Importance"] / 100,
                0.008,
                facecolor=TEAM_PRIMARY,
                edgecolor="none",
                zorder=5,
            )
        )


def recent_metric_average(df: pd.DataFrame, team_name: str, stat_col: str, n_games=None) -> float:
    team_games = (
        df[df["TEAM_NAME"] == team_name]
        .sort_values(["GAME_DATE", "GAME_ID"])
        .copy()
    )

    if n_games is not None:
        team_games = team_games.tail(n_games)

    return team_games[stat_col].mean()


def opponent_context_values(df: pd.DataFrame, opponent: str, cfg: dict) -> tuple:
    if cfg["goal"] == "Offensive Efficiency":
        season = recent_metric_average(df, opponent, "ACT_DEF_RTG")
        l10 = recent_metric_average(df, opponent, "ACT_DEF_RTG", 10)

    elif cfg["goal"] == "Defensive Efficiency":
        season = recent_metric_average(df, opponent, "ACT_OFF_RTG")
        l10 = recent_metric_average(df, opponent, "ACT_OFF_RTG", 10)

    elif cfg["goal"] == "Turnover Control":
        season = recent_metric_average(df, opponent, "ACT_OPP_TOV_PCT")
        l10 = recent_metric_average(df, opponent, "ACT_OPP_TOV_PCT", 10)

    elif cfg["goal"] == "Offensive Glass":
        season = 100 - recent_metric_average(df, opponent, "ACT_DREB_PCT")
        l10 = 100 - recent_metric_average(df, opponent, "ACT_DREB_PCT", 10)

    elif cfg["goal"] == "Free Throw Pressure":
        season = recent_metric_average(df, opponent, "ACT_OPP_FTA_RATE")
        l10 = recent_metric_average(df, opponent, "ACT_OPP_FTA_RATE", 10)

    elif cfg["goal"] == "3PT Volume":
        season = recent_metric_average(df, opponent, "ACT_OPP_3PA_RATE")
        l10 = recent_metric_average(df, opponent, "ACT_OPP_3PA_RATE", 10)

    else:
        season = recent_metric_average(df, opponent, "ACT_PACE")
        l10 = recent_metric_average(df, opponent, "ACT_PACE", 10)

    return season, l10

def advantage_color(label: str) -> str:
    if label == f"{MY_TEAM_ABBR} Edge":
        return TEAM_PRIMARY

    if label == f"{OPPONENT_ABBR} Edge":
        return OPP_PRIMARY

    return "#6B7280"

def percentile_score(values: pd.Series, value: float, higher_is_better: bool = True) -> float:
    clean_values = pd.to_numeric(values, errors="coerce").dropna()

    if len(clean_values) == 0 or pd.isna(value):
        return 50.0

    pct = 100 * (clean_values <= value).mean()

    if not higher_is_better:
        pct = 100 - pct

    return float(pct)


def latest_team_metric_values(df: pd.DataFrame, stat_col: str, n_games=None) -> pd.Series:
    rows = []

    for team_name in sorted(df["TEAM_NAME"].dropna().unique()):
        value = recent_metric_average(df, team_name, stat_col, n_games=n_games)
        rows.append({
            "TEAM_NAME": team_name,
            "Value": value,
        })

    return pd.DataFrame(rows).set_index("TEAM_NAME")["Value"]


def blended_percentile(df: pd.DataFrame, team_name: str, stat_col: str, higher_is_better: bool) -> float:
    season_values = latest_team_metric_values(df, stat_col, n_games=None)
    l10_values = latest_team_metric_values(df, stat_col, n_games=10)

    season_value = season_values.loc[team_name]
    l10_value = l10_values.loc[team_name]

    season_pct = percentile_score(season_values, season_value, higher_is_better=higher_is_better)
    l10_pct = percentile_score(l10_values, l10_value, higher_is_better=higher_is_better)

    return (0.45 * season_pct) + (0.55 * l10_pct)


def matchup_percentiles_for_goal(cfg: dict) -> tuple:
    goal = cfg["goal"]

    if goal == "Offensive Efficiency":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_OFF_RTG", higher_is_better=True)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_DEF_RTG", higher_is_better=False)

    elif goal == "Defensive Efficiency":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_DEF_RTG", higher_is_better=False)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_OFF_RTG", higher_is_better=True)

    elif goal == "Turnover Control":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_TOV_PCT", higher_is_better=False)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_OPP_TOV_PCT", higher_is_better=True)

    elif goal == "Offensive Glass":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_OREB_PCT", higher_is_better=True)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_DREB_PCT", higher_is_better=True)

    elif goal == "Free Throw Pressure":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_FTA_RATE", higher_is_better=True)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_OPP_FTA_RATE", higher_is_better=False)

    elif goal == "3PT Volume":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_3PA_RATE", higher_is_better=True)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_OPP_3PA_RATE", higher_is_better=False)

    elif goal == "Game Tempo":
        team_pct = blended_percentile(pregame_df, MY_TEAM, "ACT_PACE", higher_is_better=True)
        opp_pct = blended_percentile(pregame_df, OPPONENT, "ACT_PACE", higher_is_better=True)

    else:
        team_pct = 50.0
        opp_pct = 50.0

    return team_pct, opp_pct


def advantage_label_from_percentiles(team_pct: float, opp_pct: float, even_band: float = 10.0) -> str:
    diff = team_pct - opp_pct

    if abs(diff) <= even_band:
        return "Even"

    if diff > 0:
        return f"{MY_TEAM_ABBR} Edge"

    return f"{OPPONENT_ABBR} Edge"


def build_goal_matchup_table(goals: pd.DataFrame) -> pd.DataFrame:
    rows = []
    cfg_lookup = {cfg["goal"]: cfg for cfg in GOAL_CONFIG}

    for _, goal_row in goals.head(7).iterrows():
        cfg = cfg_lookup[goal_row["Goal"]]
        metric = cfg["metric"]

        team_season = matchup[cfg["team_avg_col"]]
        team_l10 = matchup[cfg["team_l10_col"]]
        opp_season, opp_l10 = opponent_context_values(pregame_df, OPPONENT, cfg)

        team_pct, opp_pct = matchup_percentiles_for_goal(cfg)

        rows.append({
            "Metric": metric,
            "Target": goal_row["Target"],
            f"{MY_TEAM_ABBR} Seas": format_value(team_season, metric),
            f"{MY_TEAM_ABBR} L10": format_value(team_l10, metric),
            f"{OPPONENT_ABBR} Seas": format_value(opp_season, metric),
            f"{OPPONENT_ABBR} L10": format_value(opp_l10, metric),
            "Edge": advantage_label_from_percentiles(team_pct, opp_pct),
        })

    return pd.DataFrame(rows)


def draw_goal_matchup_table(ax, df: pd.DataFrame, x, y, w, h):
    ax.add_patch(
        patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=0.8,
            edgecolor=LINE,
            facecolor=WHITE,
            zorder=1,
        )
    )

    header_h = 0.048

    ax.add_patch(
        patches.Rectangle(
            (x, y + h - header_h),
            w,
            header_h,
            facecolor=INK,
            edgecolor="none",
            zorder=2,
        )
    )

    ax.text(
        x + 0.01,
        y - 0.002 + h - header_h / 2,
        "MATCHUP METRICS",
        fontsize=14,
        color="white",
        weight="bold",
        va="center",
        zorder=4,
    )

    ax.text(
        x + w - 0.014,
        y - 0.002 + h - header_h / 2,
        "SEASON + LAST 10",
        fontsize=9.0,
        color="#D1D5DB",
        ha="right",
        va="center",
        weight="bold",
        zorder=4,
    )

    table_ax = ax.inset_axes([x + 0.004, y + 0.006, w - 0.008, h - header_h - 0.012])
    table_ax.axis("off")

    col_widths = [0.123, 0.131, 0.146, 0.114, 0.146, 0.114, 0.146]

    table = table_ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
        colWidths=col_widths,
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.08, 1.96)

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.42)

        if r == 0:
            cell.set_height(cell.get_height() * 1.25)

            if c in [2, 3]:
                cell.set_facecolor(TEAM_PRIMARY)
                cell.set_text_props(color="white", weight="bold", fontsize=9.5)
            elif c in [4, 5]:
                cell.set_facecolor(OPP_PRIMARY)
                cell.set_text_props(color="white", weight="bold", fontsize=9.5)
            elif c == 6:
                cell.set_facecolor(INK)
                cell.set_text_props(color="white", weight="bold", fontsize=9.5)
            else:
                cell.set_facecolor("#F3F4F6")
                cell.set_text_props(color=INK, weight="bold", fontsize=9.5)
            continue

        cell.set_height(cell.get_height() * 1.05)
        cell.set_facecolor("#FBFCFE" if r % 2 == 0 else WHITE)

        if c == 0:
            cell.set_text_props(color=INK, weight="bold", ha="left", fontsize=9)
        elif c == 1:
            cell.set_text_props(color=TEAM_PRIMARY, weight="bold", fontsize=9)
        elif c == 6:
            label = df.iloc[r - 1]["Edge"]
            color = advantage_color(label)
            cell.set_facecolor(color)
            cell.set_text_props(color="white", weight="bold", fontsize=9)
        else:
            cell.set_text_props(color=INK, weight="bold", fontsize=9)

    return table

def short_goal_label(goal: str) -> str:
    labels = {
        "Offensive Efficiency": "Offensive Efficiency",
        "Defensive Efficiency": "Defensive Efficiency",
        "Turnover Control": "Turnovers",
        "Offensive Glass": "O-Glass",
        "Free Throw Pressure": "FT Att. Rate",
        "3PT Volume": "3PT Volume",
        "Game Tempo": "Pace",
    }

    return labels.get(goal, goal)


def draw_goal_pressure_map(ax, goals: pd.DataFrame, x, y, w, h):
    ax.add_patch(
        patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=0.8,
            edgecolor=LINE,
            facecolor=WHITE,
            zorder=1,
        )
    )

    header_h = 0.048

    ax.add_patch(
        patches.Rectangle(
            (x, y + h - header_h),
            w,
            header_h,
            facecolor=INK,
            edgecolor="none",
            zorder=2,
        )
    )

    ax.text(
        x + 0.01,
        y - 0.002 + h - header_h / 2,
        "GOAL PRESSURE MAP",
        fontsize=14,
        color="white",
        weight="bold",
        va="center",
        zorder=4,
    )

    ax.text(
        x + w - 0.014,
        y - 0.002 + h - header_h / 2,
        "PERCENTILE EDGE",
        fontsize=9,
        color="#D1D5DB",
        ha="right",
        va="center",
        weight="bold",
        zorder=4,
    )

    cfg_lookup = {cfg["goal"]: cfg for cfg in GOAL_CONFIG}
    visual_goals = goals.head(7).copy()

    plot_x = x + 0.128
    plot_w = w - 0.178
    center_x = plot_x + plot_w / 2
    half_w = plot_w / 2

    top_y = y + h - header_h - 0.044
    row_gap = 0.041
    bar_h = 0.014

    ax.text(
        plot_x,
        y + h - header_h - 0.020,
        MY_TEAM_ABBR,
        fontsize=9.2,
        color=TEAM_PRIMARY,
        weight="bold",
        ha="left",
        va="center",
        zorder=4,
    )

    ax.text(
        center_x,
        y + h - header_h - 0.020,
        "EVEN",
        fontsize=8.1,
        color=MUTED,
        weight="bold",
        ha="center",
        va="center",
        zorder=4,
    )

    ax.text(
        plot_x + plot_w,
        y + h - header_h - 0.020,
        OPPONENT_ABBR,
        fontsize=9.2,
        color=OPP_PRIMARY,
        weight="bold",
        ha="right",
        va="center",
        zorder=4,
    )

    ax.plot(
        [center_x, center_x],
        [y + 0.040, y + h - header_h - 0.030],
        color="#CBD5E1",
        linewidth=0.9,
        zorder=3,
    )

    for i, (_, goal_row) in enumerate(visual_goals.iterrows()):
        cfg = cfg_lookup[goal_row["Goal"]]
        team_pct, opp_pct = matchup_percentiles_for_goal(cfg)
        diff = team_pct - opp_pct
        edge_label = advantage_label_from_percentiles(team_pct, opp_pct)

        row_y = top_y - i * row_gap

        ax.text(
            x + 0.014,
            row_y,
            short_goal_label(goal_row["Goal"]),
            fontsize=9.2,
            color=INK,
            weight="bold",
            ha="left",
            va="center",
            zorder=4,
        )

        ax.add_patch(
            patches.Rectangle(
                (plot_x, row_y - bar_h / 2),
                plot_w,
                bar_h,
                facecolor="#EEF2F7",
                edgecolor="none",
                zorder=2,
            )
        )

        scaled_diff = max(min(abs(diff), 60), 3) / 60
        bar_w = half_w * scaled_diff

        if diff >= 0:
            bar_x = center_x - bar_w
            value_label = f"+{diff:.0f}" if edge_label != "Even" else f"+{diff:.0f}"
        else:
            bar_x = center_x
            value_label = f"{diff:.0f}"

        if edge_label == f"{MY_TEAM_ABBR} Edge":
            color = TEAM_PRIMARY
        elif edge_label == f"{OPPONENT_ABBR} Edge":
            color = OPP_PRIMARY
        else:
            color = "#9CA3AF"

        ax.add_patch(
            patches.Rectangle(
                (bar_x, row_y - bar_h / 2),
                bar_w,
                bar_h,
                facecolor=color,
                edgecolor="none",
                zorder=4,
            )
        )

        ax.text(
            plot_x + plot_w + 0.010,
            row_y,
            value_label,
            fontsize=8.6,
            color=color,
            weight="bold",
            ha="left",
            va="center",
            zorder=4,
        )

    ax.text(
        x + 0.014,
        y + 0.020,
        "Bars show blended season/L10 percentile gap. Within 10 points = even.",
        fontsize=6.7,
        color=MUTED,
        ha="left",
        va="center",
        zorder=4,
    )

def draw_visual_placeholder(ax, x, y, w, h):
    ax.add_patch(
        patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=0.8,
            edgecolor=LINE,
            facecolor=WHITE,
            zorder=1,
        )
    )

    ax.add_patch(
        patches.Rectangle(
            (x, y + h - 0.030),
            w,
            0.030,
            facecolor="#F3F4F6",
            edgecolor="none",
            zorder=2,
        )
    )

    ax.text(
        x + 0.014,
        y + h - 0.015,
        "VISUAL SLOT",
        fontsize=9.2,
        color=INK,
        weight="bold",
        va="center",
        zorder=4,
    )

    ax.text(
        x + w - 0.014,
        y + h - 0.015,
        "next step",
        fontsize=7.5,
        color=MUTED,
        ha="right",
        va="center",
        weight="bold",
        zorder=4,
    )


def draw_one_page_report():
    fig = plt.figure(figsize=(14, 8.5), facecolor=PAGE_BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    draw_title_bar(ax)

    my_profile = team_profiles.loc[team_profiles["TEAM_NAME"] == MY_TEAM].iloc[0]
    opp_profile = team_profiles.loc[team_profiles["TEAM_NAME"] == OPPONENT].iloc[0]

    draw_team_header_block(
        fig,
        ax,
        0.035,
        0.735,
        0.455,
        0.155,
        MY_TEAM,
        MY_TEAM_ABBR,
        TEAM_PRIMARY,
        TEAM_SECONDARY,
        my_profile,
        team_logo_img,
    )

    draw_team_header_block(
        fig,
        ax,
        0.510,
        0.735,
        0.455,
        0.155,
        OPPONENT,
        OPPONENT_ABBR,
        OPP_PRIMARY,
        OPP_SECONDARY,
        opp_profile,
        opp_logo_img,
    )

    draw_matchup_focus_banner(ax)
    draw_goal_cards(ax, gameplan_goals)

    matchup_table = build_goal_matchup_table(gameplan_goals)

    draw_goal_matchup_table(
        ax,
        matchup_table,
        0.035,
        0.075,
        0.535,
        0.400,
    )

    draw_goal_pressure_map(
        ax,
        gameplan_goals,
        0.590,
        0.075,
        0.375,
        0.400,
    )

    fig.text(
        0.035,
        0.025,
        "Pregame-known inputs only. Goals blend team baseline, recent form, opponent context, winning thresholds, and model signal weight.",
        fontsize=7.8,
        color=MUTED,
    )

    return fig

## 15. Export PDF

In [1890]:
REPORT_PATH = OUTPUT_DIR / f"{MY_TEAM_ABBR}_vs_{OPPONENT_ABBR}_{SEASON}_{SEASON_TYPE.replace(' ', '_')}_gameplan.pdf"

fig = draw_one_page_report()
fig.savefig(REPORT_PATH, format="pdf", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)

print(f"Saved one-page report: {REPORT_PATH}")

Saved one-page report: pregame_gameplan_reports/NYK_vs_BOS_2024-25_Regular_Season_gameplan.pdf
